# 00b — E-Distance Tutorial

**Dataset:** `NormanWeissman2019` (scPerturb-harmonized, ~111K cells, K562 CRISPRa screen)  
**Accession:** Zenodo 7041849  
**Phase:** 1  
**Date:** 2026-03-11  
**Objective:** Understand and implement E-distance / E-test for quantifying perturbation effect size
as described in the scPerturb paper (Peidli et al., *Nature Methods* 2024).  
We build the metric from first principles, verify it on simulated data, and apply it to
a real CRISPR screen to rank perturbations by effect size.

## Table of Contents

1. [Background: What is E-distance?](#background)  
   1.1 [Why a distributional metric?](#distributional)  
   1.2 [The energy distance formula](#formula)  
   1.3 [Key properties and comparisons](#properties)  
2. [Building intuition with 2-D simulations](#intuition)  
3. [Manual implementation from scratch](#manual)  
4. [Preprocessing: normalise, HVG, PCA](#preprocessing)  
5. [Computing E-distances for all perturbations](#edistances)  
6. [The E-test: permutation-based significance](#etest)  
   6.1 [Algorithm and test statistic](#algorithm)  
   6.2 [Multiple-testing correction (BH / FDR)](#fdr)  
7. [Using pertpy's Distance module](#pertpy)  
8. [Visualisation: rankings and volcano plot](#viz)  
9. [QC: cell-count requirements](#qc)  
10. [Summary and key takeaways](#summary)

## 0. Setup

In [ ]:
import anndata as ad
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
import boto3
from pathlib import Path
from scipy.spatial.distance import cdist
from statsmodels.stats.multitest import multipletests
import warnings
warnings.filterwarnings('ignore')

sc.settings.verbosity = 1
sc.settings.set_figure_params(dpi=100, frameon=False)
np.random.seed(42)

DATA_DIR   = Path('../../data/raw/scperturb')
DATA_DIR.mkdir(parents=True, exist_ok=True)
RESULTS    = Path('../../results')
FIG_DIR    = RESULTS / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)
S3_BUCKET  = 'learn-perturb-seq'
H5AD_FILE  = 'NormanWeissman2019_filtered.h5ad'
LOCAL_PATH = DATA_DIR / H5AD_FILE

print('Setup complete.')

In [ ]:
# Download from S3 if not already cached (notebook 00a likely did this already)
if not LOCAL_PATH.exists():
    print(f'Downloading {H5AD_FILE} ...')
    s3 = boto3.client('s3')
    s3.download_file(S3_BUCKET, f'raw/scperturb/{H5AD_FILE}', str(LOCAL_PATH))
    print('Done.')
else:
    print(f'Using cached file: {LOCAL_PATH}')

adata = sc.read_h5ad(LOCAL_PATH)
print(adata)
print(f'\nTotal cells:           {len(adata):,}')
print(f'Unique perturbations:  {adata.obs["perturbation"].nunique()}')
print(f'Control cells:         {(adata.obs["perturbation"] == "control").sum():,}')

<a id='background'></a>
## 1. Background: What is E-distance?

<a id='distributional'></a>
### 1.1  Why do we need a distributional metric?

In a Perturb-seq experiment each perturbation (gene knock-out, CRISPRa activation, drug treatment)
shifts a *population* of cells through gene-expression space.  We want to quantify:

1. **Effect size** — how strongly does perturbation X move cells relative to control?
2. **Significance** — is the shift larger than sampling noise?
3. **Similarity** — which perturbations produce similar transcriptional programs?

The simplest approach — comparing **mean expression vectors** — misses critical information:

| Scenario | Mean shift | Distribution shift |
|----------|-----------|--------------------|
| Perturbation moves all cells in one direction | Large | Large |
| Perturbation *spreads* cells without moving the mean | **Zero** | **Large** |
| Perturbation splits cells into two sub-populations | Small | **Large** |

We need a metric that operates on **distributions**, not just means.  
The **E-distance** (energy distance) is the metric used by the scPerturb paper precisely because
it captures the *full* distributional shift — location, spread, and shape.

---

<a id='formula'></a>
### 1.2  The energy distance formula

Let $X \sim P$ (perturbed cells) and $Y \sim Q$ (control cells) be random vectors in
$\mathbb{R}^d$.  Define $X'$ as an independent copy of $X$, and $Y'$ as an independent copy of $Y$.
The **energy distance** between distributions $P$ and $Q$ is:

$$E(P, Q) = 2\,\mathbb{E}[\|X - Y\|] - \mathbb{E}[\|X - X'\|] - \mathbb{E}[\|Y - Y'\|]$$

where $\|\cdot\|$ is the Euclidean norm (L2 distance).

**Breaking down the three terms:**

- **Term 1: $2\,\mathbb{E}[\|X - Y\|]$** — twice the average distance from a perturbed cell to a
  control cell.  This is large when the two clouds are far apart.

- **Term 2: $\mathbb{E}[\|X - X'\|]$** — average pairwise distance *within* the perturbed group.
  This adjusts for how spread out the perturbed population is.

- **Term 3: $\mathbb{E}[\|Y - Y'\|]$** — average pairwise distance *within* the control group.
  This adjusts for the baseline spread of the control population.

**Intuition:** E-distance is large when the two clouds are *farther from each other* than they
are *internally spread*.  It equals zero if and only if $P = Q$.

**Sample estimator (what we compute in practice):**

Given $n$ perturbed cells $\{x_1, \ldots, x_n\}$ and $m$ control cells $\{y_1, \ldots, y_m\}$:

$$\hat{E}(P, Q) =
  \frac{2}{nm}\sum_{i=1}^{n}\sum_{j=1}^{m}\|x_i - y_j\|
  - \frac{1}{n^2}\sum_{i=1}^{n}\sum_{k=1}^{n}\|x_i - x_k\|
  - \frac{1}{m^2}\sum_{j=1}^{m}\sum_{l=1}^{m}\|y_j - y_l\|$$

Which simplifies to:

$$\hat{E} = 2 \cdot \text{mean}(XY\text{ distances}) - \text{mean}(XX\text{ distances}) - \text{mean}(YY\text{ distances})$$

---

<a id='properties'></a>
### 1.3  Key properties and comparisons to other metrics

| Property | E-distance | Wasserstein | MMD | L2 of means |
|----------|-----------|-------------|-----|------------|
| Non-negative | ✓ | ✓ | ✓ | ✓ |
| Zero iff same distribution | ✓ | ✓ | ✓* | ✗ |
| Detects spread differences | ✓ | ✓ | ✓ | ✗ |
| Sample estimator | Closed-form O(n²) | Optimal transport O(n³) | Kernel, O(n²) | O(n) |
| High-dim stability | Good | Suffers curse-of-dim | Kernel choice | Poor |

\* With a universal kernel.

**Why PCA space?**  
Computing pairwise Euclidean distances in 33,000-dimensional gene expression space is:
1. Computationally prohibitive — 33k multiplications per cell pair
2. Dominated by noise genes that carry little biological signal

We project to **50 PCA components** (typically capturing > 60% of variance) before computing
E-distances.  This follows the scPerturb paper's convention and yields distances that reflect
biologically meaningful co-variation rather than technical noise.

**Theoretical origin:**  
Energy distance was introduced by Székely & Rizzo (2004, *InterStat*) as the multivariate
generalisation of the Cramér-von Mises statistic.  It is also equal to the squared MMD with a
negative definite kernel $k(x,y) = -\|x-y\|$, which is the source of its theoretical guarantees.

<a id='intuition'></a>
## 2. Building intuition with 2-D simulations

Before touching any scRNA-seq data, let's verify our understanding using synthetic 2-D clouds
where we fully control the true underlying distributions.  We will simulate four scenarios:

| Scenario | Effect | Expected E-distance |
|----------|--------|--------------------|
| A — Identical distributions | None | ≈ 0 |
| B — Shifted mean only | Location shift | Moderate–large |
| C — Increased spread only | Scale change | Moderate |
| D — Bimodal split | Shape change | Large |

In [ ]:
def edist_2d(X, Y):
    """Compute E-distance between two sample arrays (rows = observations)."""
    XY = cdist(X, Y, metric='euclidean').mean()
    XX = cdist(X, X, metric='euclidean').mean()
    YY = cdist(Y, Y, metric='euclidean').mean()
    return 2 * XY - XX - YY


rng = np.random.default_rng(42)
n = 300  # cells per group

# Reference (control) cloud — isotropic Gaussian centred at origin
ctrl = rng.multivariate_normal([0, 0], [[1, 0], [0, 1]], size=n)

scenarios = {
    'A: identical'    : rng.multivariate_normal([0, 0], [[1, 0], [0, 1]], size=n),
    'B: mean shift'   : rng.multivariate_normal([2, 2], [[1, 0], [0, 1]], size=n),
    'C: spread only'  : rng.multivariate_normal([0, 0], [[4, 0], [0, 4]], size=n),
    'D: bimodal split': np.vstack([
        rng.multivariate_normal([ 2, 2], [[0.5, 0], [0, 0.5]], size=n//2),
        rng.multivariate_normal([-2,-2], [[0.5, 0], [0, 0.5]], size=n//2),
    ]),
}

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, (name, pert) in zip(axes, scenarios.items()):
    e = edist_2d(pert, ctrl)
    ax.scatter(*ctrl.T, s=10, alpha=0.4, color='steelblue', label='control')
    ax.scatter(*pert.T, s=10, alpha=0.4, color='tomato',    label='perturbed')
    ax.set_title(f'{name}\nE-dist = {e:.3f}', fontsize=9)
    ax.set_xlim(-6, 8); ax.set_ylim(-6, 8)
    ax.legend(fontsize=7)

plt.suptitle('E-distance across four perturbation scenarios (2-D)', y=1.02)
plt.tight_layout()
plt.savefig(FIG_DIR / '00b_edist_2d_intuition.pdf', bbox_inches='tight')
plt.show()

print('Scenario              E-distance')
print('-' * 35)
for name, pert in scenarios.items():
    print(f'{name:22s}  {edist_2d(pert, ctrl):.4f}')

**Interpretation:**

- **Scenario A** (identical): E-distance ≈ 0 — both clouds were sampled from the same distribution,
  so the cross-group mean distance equals the within-group spread. ✓
- **Scenario B** (mean shift): E-distance is clearly positive — the cross-group distances are larger
  than the within-group spread.
- **Scenario C** (spread only): E-distance > 0 even though the **means are identical**.  
  This demonstrates that E-distance detects variance changes that a mean-comparison would miss.
- **Scenario D** (bimodal): Large E-distance — the bimodal cloud is far from the unimodal control
  in a distributional sense.

The negative part of the formula ($-XX - YY$) subtracts the *baseline spread* of each population,
ensuring the metric rewards genuine between-group separation rather than punishing tight distributions.

<a id='manual'></a>
## 3. Manual implementation from scratch

Below is a self-contained, heavily annotated implementation.  This is what `src/perturbation_analysis/edistance.py` wraps.  Reading through it line-by-line cements the connection between the mathematical formula and the code.

In [ ]:
def compute_edist(X_pert: np.ndarray, X_ctrl: np.ndarray) -> float:
    """
    Compute the sample E-distance between a perturbed population and a control population.

    Parameters
    ----------
    X_pert : (n, d) array — coordinates of n perturbed cells in d-dimensional space
    X_ctrl : (m, d) array — coordinates of m control cells in d-dimensional space

    Returns
    -------
    float — E-distance; always >= 0; equals 0 iff the two samples are identical

    Formula
    -------
    E(P, Q) = 2 * mean(||x_i - y_j||)        [cross-group distances]
              - mean(||x_i - x_k||)           [within-perturbation spread]
              - mean(||y_j - y_l||)           [within-control spread]

    Notes
    -----
    - `cdist(A, B)` returns an (n_A, n_B) matrix of all pairwise Euclidean distances.
    - `cdist(X, X)` includes the diagonal (self-distances = 0).  For large n the bias
      introduced by these zeros is negligible (factor n/(n-1)), so we keep them for
      simplicity and consistency with the scPerturb reference implementation.
    - Computation is O(n*m*d) for the cross-group term — the dominant cost.
      For n=m=1000 cells in d=50 PCA dimensions this is ~50M multiplications.
    """
    # --- Term 1: cross-group mean distance -----------------------------------
    # cdist shape: (n_pert, n_ctrl)
    # .mean() averages over ALL n*m cell pairs
    cross_mean = cdist(X_pert, X_ctrl, metric='euclidean').mean()

    # --- Term 2: within-perturbation mean distance ---------------------------
    # cdist shape: (n_pert, n_pert)
    # Includes the zero diagonal: d(x_i, x_i) = 0
    within_pert_mean = cdist(X_pert, X_pert, metric='euclidean').mean()

    # --- Term 3: within-control mean distance --------------------------------
    within_ctrl_mean = cdist(X_ctrl, X_ctrl, metric='euclidean').mean()

    # --- Combine: 2*cross - within_pert - within_ctrl -----------------------
    edistance = 2.0 * cross_mean - within_pert_mean - within_ctrl_mean

    # E-distance is always >= 0 in theory; small negative values can arise from
    # floating-point rounding when the two samples are very similar.
    return max(edistance, 0.0)


def compute_estatistic(X_pert: np.ndarray, X_ctrl: np.ndarray) -> float:
    """
    Compute the E-statistic (sample-size-scaled E-distance) used as the test statistic
    for the E-test.

    E_statistic = (n * m) / (2 * (n + m)) * E-distance

    This scaling makes the statistic comparable across permutations with different
    group sizes and is consistent with the energy statistics literature
    (Székely & Rizzo 2004, 2013).
    """
    n, m = len(X_pert), len(X_ctrl)
    e = compute_edist(X_pert, X_ctrl)
    return (n * m) / (2 * (n + m)) * e


# Quick sanity check --------------------------------------------------------
rng = np.random.default_rng(0)
same   = rng.standard_normal((200, 10))        # identical distributions
ctrl_  = rng.standard_normal((200, 10))
shifted = ctrl_ + 3                             # large mean shift

e_same    = compute_edist(same, rng.standard_normal((200, 10)))
e_shifted = compute_edist(shifted, rng.standard_normal((200, 10)))

print(f'E-distance (same distribution, n=200 each): {e_same:.4f}   <- expect ≈ 0')
print(f'E-distance (mean shift = 3,    n=200 each): {e_shifted:.4f}  <- expect large')

<a id='preprocessing'></a>
## 4. Preprocessing: normalise, HVG selection, PCA

E-distances must be computed in a low-noise, informative representation.  The standard Perturb-seq
preprocessing pipeline produces a 50-PC embedding that we will use throughout:

1. **Filter** to single-perturbation cells (`nperts == 1`) — double perturbations are a separate
   analysis unit (see Norman 2019 combinatorial screen notebooks).
2. **Normalise** counts per cell to 10,000 UMIs (library-size normalisation).
3. **Log-transform** with log(1 + x) to stabilise variance across expression levels.
4. **Select** the 2,000 most highly variable genes (HVGs) — these capture biological variation
   while discarding housekeeping genes that add noise.
5. **Scale** each gene to zero mean / unit variance before PCA (so high-expression genes
   do not dominate principal components).
6. **PCA** to 50 components — the representation used for all subsequent distance computations.

In [ ]:
# ---- 4.1  Filter to single perturbations -----------------------------------
adata_s = adata[adata.obs['nperts'] == 1].copy()
print(f'Cells with exactly 1 perturbation: {len(adata_s):,}')
print(f'Perturbations remaining:           {adata_s.obs["perturbation"].nunique()}')

# ---- 4.2  Log-normalise ----------------------------------------------------
# Store raw counts in .raw so we can recover them later
adata_s.raw = adata_s.copy()
sc.pp.normalize_total(adata_s, target_sum=1e4)    # library-size normalise to 10K
sc.pp.log1p(adata_s)                               # log(1 + x)
print('\nNormalisation complete. Value range (sample):', adata_s.X[:5, :5].toarray().round(2))

# ---- 4.3  Highly variable genes --------------------------------------------
N_HVG = 2000
sc.pp.highly_variable_genes(adata_s, n_top_genes=N_HVG, subset=True)
print(f'\nHVG selection: kept {N_HVG} most variable genes (from {adata.n_vars:,})')

# ---- 4.4  Scale (zero mean, unit variance per gene) -------------------------
# NOTE: sc.pp.scale computes (x - mean) / std per gene. Any gene with std = 0
# in this cell subset (constant expression across all cells) produces 0/0 = NaN.
# These NaN values propagate through PCA — ANY linear combination containing a
# NaN column becomes NaN — so every PCA coordinate for every cell becomes NaN,
# which makes all E-distances NaN.
# Fix: replace NaN with 0 after scaling. A zero-variance gene scaled to 0 is
# semantically correct: it is at its mean and contributes no discriminating
# information to the PCA, so 0 is the right fill value.
sc.pp.scale(adata_s, max_value=10)   # clip outliers at 10 SD

import scipy.sparse as sp
X_arr = adata_s.X.toarray() if sp.issparse(adata_s.X) else adata_s.X
n_nan_genes = np.isnan(X_arr).any(axis=0).sum()
n_nan_vals  = np.isnan(X_arr).sum()
if n_nan_vals > 0:
    print(f'\n⚠  {n_nan_genes} zero-variance genes produced {n_nan_vals:,} NaN values after scaling.')
    print('   Replacing NaN → 0 (gene is at its mean; contributes nothing to PCA).')
    X_arr = np.nan_to_num(X_arr, nan=0.0)
    adata_s.X = sp.csr_matrix(X_arr) if sp.issparse(adata_s.X) else X_arr
else:
    print('\nNo NaN values after scaling. ✓')

# ---- 4.5  PCA ---------------------------------------------------------------
N_PCA = 50
sc.tl.pca(adata_s, n_comps=N_PCA, svd_solver='arpack')
print(f'PCA complete. adata_s.obsm["X_pca"] shape: {adata_s.obsm["X_pca"].shape}')

# Final sanity check: confirm no NaN in PCA coordinates
n_nan_pca = np.isnan(adata_s.obsm['X_pca']).sum()
print(f'NaN values in X_pca: {n_nan_pca}  {"✓" if n_nan_pca == 0 else "✗ — investigate further"}')

# Variance explained
var_explained = adata_s.uns['pca']['variance_ratio']
print(f'Variance explained by first 10 PCs:  {var_explained[:10].sum():.1%}')
print(f'Variance explained by all {N_PCA} PCs:    {var_explained.sum():.1%}')

In [ ]:
# Scree plot — confirms that 50 PCs capture the majority of variance
fig, ax = plt.subplots(figsize=(7, 3))
ax.bar(range(1, N_PCA + 1), var_explained * 100, color='steelblue', width=0.8)
ax.axvline(10, color='orange', ls='--', lw=1, label='PC 10')
ax.set_xlabel('Principal component')
ax.set_ylabel('Variance explained (%)')
ax.set_title('Scree plot — NormanWeissman2019 (HVG subset)')
ax.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / '00b_scree_plot.pdf', bbox_inches='tight')
plt.show()

<a id='edistances'></a>
## 5. Computing E-distances for all perturbations

We compute the E-distance from each perturbation to the control population in PCA space.
**Minimum cell-count requirement:** The scPerturb paper recommends ≥ 200 cells per perturbation
for reliable E-distance estimates.  Below 200, the within-group distance estimate is noisy
and E-distances can be spuriously inflated.  We flag perturbations with < 200 cells.

**Computational strategy:**  
With ~11,000 control cells, `cdist(X_pert, X_ctrl)` would produce an 11K × 11K matrix for
the within-control term — 484M elements at float64.  To keep memory tractable, we
**subsample the control group** to 2,000 cells (still 10× more than most perturbation groups),
which reduces the within-control matrix to 2K × 2K = 4M elements while barely affecting
the distance estimate (stable to < 1% error with n > 500).

In [ ]:
# ---- 5.1  Extract PCA coordinates ------------------------------------------
X_pca   = adata_s.obsm['X_pca']           # (n_cells, 50)
labels  = adata_s.obs['perturbation'].values

# Subsample control to 2000 cells for computational efficiency
CTRL_SUBSAMPLE = 2000
ctrl_idx = np.where(labels == 'control')[0]
rng      = np.random.default_rng(42)
ctrl_idx_sub = rng.choice(ctrl_idx, size=min(CTRL_SUBSAMPLE, len(ctrl_idx)), replace=False)
X_ctrl   = X_pca[ctrl_idx_sub]            # (2000, 50)
print(f'Control cells used for E-distance computation: {len(X_ctrl):,}')

# ---- 5.2  Perturbation cell counts -----------------------------------------
pert_counts = adata_s.obs['perturbation'].value_counts()
pert_counts = pert_counts[pert_counts.index != 'control']

MIN_CELLS = 200
pert_pass = pert_counts[pert_counts >= MIN_CELLS].index.tolist()
pert_warn = pert_counts[pert_counts <  MIN_CELLS].index.tolist()
print(f'\nPerturbations with >= {MIN_CELLS} cells: {len(pert_pass)}')
print(f'Perturbations with  < {MIN_CELLS} cells: {len(pert_warn)}  (flagged, still computed)')

# ---- 5.3  Compute E-distances for all single perturbations -----------------
edist_results = {}
for pert in pert_counts.index:
    pert_idx = np.where(labels == pert)[0]
    X_pert   = X_pca[pert_idx]
    edist_results[pert] = compute_edist(X_pert, X_ctrl)

edist_series = pd.Series(edist_results, name='edistance').sort_values(ascending=False)
print(f'\nE-distances computed for {len(edist_series)} perturbations.')
print(f'\nTop 10 perturbations by E-distance:')
print(edist_series.head(10).round(4))

In [ ]:
# Summary statistics
print('E-distance summary across all perturbations:')
print(edist_series.describe().round(4))
print(f'\nPerturbations with E-dist > 1.0:  {(edist_series > 1.0).sum()}')
print(f'Perturbations with E-dist > 0.5:  {(edist_series > 0.5).sum()}')
print(f'Perturbations with E-dist ≈ 0:    {(edist_series < 0.05).sum()}')

# Mark low-cell-count perturbations
edist_series_flagged = edist_series.copy()
is_low_cells = edist_series.index.isin(pert_warn)

<a id='etest'></a>
## 6. The E-test: permutation-based significance testing

<a id='algorithm'></a>
### 6.1  Algorithm and test statistic

A large E-distance could reflect a genuine perturbation effect — or it could arise from chance
sampling variation, especially when a perturbation group has few cells.  The **E-test** assigns
a p-value to each E-distance via a **permutation test**.

**Null hypothesis:** The perturbed cells and control cells are drawn from the *same*
distribution — i.e., the perturbation had no transcriptional effect.

**Algorithm:**

```
1.  Compute T_obs = E-statistic(X_pert, X_ctrl)         ← observed test stat

2.  Pool all cells: Z = concatenate(X_pert, X_ctrl)     ← n + m cells total

3.  For b = 1 ... B:
        shuffle row order of Z randomly
        Z_A = Z[:n]    ← pseudo-perturbation group (same size as original)
        Z_B = Z[n:]    ← pseudo-control group
        T_b = E-statistic(Z_A, Z_B)

4.  p-value = (number of b where T_b >= T_obs) / B
```

**The E-statistic (test statistic):**

We use the sample-size-scaled version:

$$T = \frac{n \cdot m}{2(n + m)} \cdot \hat{E}(P, Q)$$

This scaling makes permuted and observed statistics comparable when group sizes differ
across permutations (they never differ here, but it is the conventional form).

**Why permutation and not a parametric test?**  
Single-cell gene expression is not multivariate-Gaussian (it is overdispersed, zero-inflated,
and high-dimensional).  Permutation tests are distribution-free and remain valid under any
null distribution.  The only assumption is **exchangeability** under the null, which holds
if cells are randomly assigned to perturbation / control.

**Number of permutations:**  
Use B = 1,000 for exploratory analysis (p-value resolution = 1/1000).  
Use B = 10,000 for final publication-quality results (p-value resolution = 1/10,000).  
For B = 1000 and true null, the standard error of the p-value estimate is
$\sqrt{p(1-p)/B} \approx 0.016$ at p = 0.05 — sufficient for screening.

<a id='fdr'></a>
### 6.2  Multiple-testing correction (Benjamini-Hochberg / FDR)

When testing 100+ perturbations simultaneously, we expect ~5 false positives at α = 0.05 by
chance alone.  We apply the **Benjamini-Hochberg procedure** to control the **False Discovery
Rate (FDR)** at 5%.  This is less conservative than Bonferroni correction (which controls the
family-wise error rate) and is standard for Perturb-seq screens.

In [ ]:
def run_etest(
    X_pert: np.ndarray,
    X_ctrl: np.ndarray,
    n_perms: int = 1000,
    rng: np.random.Generator = None,
) -> dict:
    """
    Permutation-based E-test for equality of two multivariate distributions.

    Parameters
    ----------
    X_pert   : (n, d) array — perturbed cell PCA coordinates
    X_ctrl   : (m, d) array — control cell PCA coordinates
    n_perms  : number of permutations (1000 for exploration, 10000 for final results)
    rng      : numpy random generator (for reproducibility)

    Returns
    -------
    dict with keys:
        'edistance'   : observed E-distance
        'estatistic'  : observed E-statistic (sample-size scaled)
        'pvalue'      : permutation p-value (1-sided, H1: E-dist > 0)
        'null_dist'   : array of permuted E-statistics (the null distribution)

    Notes
    -----
    Computational cost: O(n_perms * n * m * d).  For n=m=300, d=50, n_perms=1000
    this is ~4.5B multiplications — a few seconds in Python.  For large groups,
    subsample to ~500 cells before calling this function.
    """
    if rng is None:
        rng = np.random.default_rng()

    n = len(X_pert)
    m = len(X_ctrl)

    # --- Observed test statistic ---
    e_obs  = compute_edist(X_pert, X_ctrl)
    t_obs  = (n * m) / (2 * (n + m)) * e_obs

    # --- Pool all cells for permutation ---
    Z = np.vstack([X_pert, X_ctrl])   # (n + m, d)

    # --- Permutation loop ---
    null_stats = np.empty(n_perms)
    for b in range(n_perms):
        # Shuffle the combined pool and split into two groups of the same sizes
        perm = rng.permutation(n + m)
        Z_A  = Z[perm[:n]]    # pseudo-perturbation (same n as observed)
        Z_B  = Z[perm[n:]]    # pseudo-control (same m as observed)
        e_b  = compute_edist(Z_A, Z_B)
        null_stats[b] = (n * m) / (2 * (n + m)) * e_b

    # --- p-value: fraction of permuted stats >= observed ---
    # Add 1 to numerator and denominator (Phipson & Smyth 2010 correction
    # to avoid p-value = 0 with finite permutations)
    pvalue = (np.sum(null_stats >= t_obs) + 1) / (n_perms + 1)

    return {
        'edistance' : e_obs,
        'estatistic': t_obs,
        'pvalue'    : pvalue,
        'null_dist' : null_stats,
    }

print('E-test function defined.')

In [ ]:
# Run the E-test on perturbations with >= MIN_CELLS cells.
# To keep this tutorial fast we use n_perms=1000 and subsample large groups.

N_PERMS       = 1000   # increase to 10000 for final analysis
MAX_PERT_SUBSAMPLE = 500   # subsample large perturbation groups for speed
rng_etest = np.random.default_rng(99)

etest_results = {}

for pert in pert_pass:   # only perturbations with >= 200 cells
    pert_idx = np.where(labels == pert)[0]
    X_pert   = X_pca[pert_idx]

    # Subsample perturbation group if very large (> MAX_PERT_SUBSAMPLE)
    if len(X_pert) > MAX_PERT_SUBSAMPLE:
        sub = rng_etest.choice(len(X_pert), size=MAX_PERT_SUBSAMPLE, replace=False)
        X_pert = X_pert[sub]

    # Subsample control to same size as perturbation for balanced test
    ctrl_sub_idx = rng_etest.choice(len(X_ctrl), size=min(MAX_PERT_SUBSAMPLE, len(X_ctrl)), replace=False)
    X_ctrl_sub   = X_ctrl[ctrl_sub_idx]

    result = run_etest(X_pert, X_ctrl_sub, n_perms=N_PERMS, rng=rng_etest)
    etest_results[pert] = result

print(f'E-test complete for {len(etest_results)} perturbations.')

# Collect into a DataFrame
df_etest = pd.DataFrame({
    'perturbation': list(etest_results.keys()),
    'edistance'  : [r['edistance']  for r in etest_results.values()],
    'estatistic' : [r['estatistic'] for r in etest_results.values()],
    'pvalue'     : [r['pvalue']     for r in etest_results.values()],
    'n_cells'    : [pert_counts[p]  for p in etest_results.keys()],
}).sort_values('edistance', ascending=False).reset_index(drop=True)

print(df_etest.head(10).to_string(index=False))

In [ ]:
# ---- 6.2  Benjamini-Hochberg FDR correction --------------------------------

reject, pvalue_adj, _, _ = multipletests(
    df_etest['pvalue'].values,
    alpha=0.05,
    method='fdr_bh'     # Benjamini-Hochberg
)

df_etest['pvalue_adj'] = pvalue_adj
df_etest['significant'] = reject

n_sig = reject.sum()
print(f'Perturbations with FDR-adjusted p < 0.05:  {n_sig} / {len(df_etest)}')
print(f'Fraction significant: {n_sig/len(df_etest):.1%}')

# Null distribution plot for one example perturbation
example_pert = df_etest.iloc[0]['perturbation']   # highest E-distance
null_dist    = etest_results[example_pert]['null_dist']
t_obs_ex     = etest_results[example_pert]['estatistic']
p_ex         = etest_results[example_pert]['pvalue']

fig, ax = plt.subplots(figsize=(7, 3))
ax.hist(null_dist, bins=40, color='lightgray', edgecolor='white', label='Null (permuted)')
ax.axvline(t_obs_ex, color='tomato', lw=2, label=f'Observed T = {t_obs_ex:.2f}')
ax.fill_betweenx(
    [0, ax.get_ylim()[1] if ax.get_ylim()[1] > 0 else 10],
    t_obs_ex, null_dist.max() * 1.05,
    alpha=0.15, color='tomato', label=f'p = {p_ex:.4f}'
)
ax.set_xlabel('E-statistic (permuted)')
ax.set_ylabel('Count')
ax.set_title(f'Null distribution for {example_pert}\n(B = {N_PERMS} permutations)')
ax.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / f'00b_null_dist_{example_pert}.pdf', bbox_inches='tight')
plt.show()
print(f'\n{example_pert}: E-distance = {etest_results[example_pert]["edistance"]:.4f}, p = {p_ex:.4f}')

<a id='pertpy'></a>
## 7. Using pertpy's Distance module

The manual implementation above is instructive, but for production analysis use
`pertpy.tools.Distance` which provides:

- Multiple distance metrics beyond E-distance (Wasserstein, MMD, pseudobulk distances, ...)
- Optimised pairwise computation
- Built-in permutation testing
- Consistent API across all metrics

The `pertpy` implementation computes **pairwise** distances between *all* perturbation groups,
giving a full distance matrix (perturbations × perturbations) rather than just distances to
control.  This is useful for clustering perturbations by their transcriptional profiles.

In [ ]:
import pertpy as pt

# ---- 7.1  Create a small subset for demonstration -------------------------
# Use only perturbations with >= 200 cells + control, to keep computation tractable
keep_perts = pert_pass + ['control']
adata_sub  = adata_s[adata_s.obs['perturbation'].isin(keep_perts)].copy()
print(f'Subset: {len(adata_sub):,} cells, {adata_sub.obs["perturbation"].nunique()} perturbation groups')

# ---- 7.2  Initialise Distance with E-distance metric ----------------------
# obsm_key='X_pca' tells pertpy to use the PCA embedding we computed above.
# The metric='edistance' option maps to the same formula as our manual implementation.
distance = pt.tl.Distance(metric='edistance', obsm_key='X_pca')

# ---- 7.3  Compute all perturbation vs. control distances -------------------
# distance.onesided_distances computes each perturbation vs. control
# groupby='perturbation' uses adata.obs['perturbation'] as group labels
pt_edist = distance.onesided_distances(
    adata_sub,
    groupby='perturbation',
    baseline='control',
    show_progressbar=True,
)

print('\npertpy E-distances (top 10):')
print(pt_edist.sort_values(ascending=False).head(10).round(4))

In [ ]:
# ---- 7.4  Compare pertpy results vs manual implementation ------------------
# Both should give the same E-distances (minor floating-point differences expected)

shared_perts = set(pt_edist.index) & set(edist_series.index)
compare_df = pd.DataFrame({
    'manual' : edist_series[list(shared_perts)],
    'pertpy' : pt_edist[list(shared_perts)],
}).dropna()

corr = compare_df.corr(method='pearson').loc['manual', 'pertpy']
print(f'Pearson r between manual and pertpy E-distances: {corr:.6f}')

fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(compare_df['manual'], compare_df['pertpy'], s=25, alpha=0.6, color='steelblue')
lims = [min(compare_df.min()), max(compare_df.max())]
ax.plot(lims, lims, 'r--', lw=1, label='y = x')
ax.set_xlabel('Manual E-distance')
ax.set_ylabel('pertpy E-distance')
ax.set_title(f'Manual vs pertpy results\nr = {corr:.4f}')
ax.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / '00b_manual_vs_pertpy.pdf', bbox_inches='tight')
plt.show()

<a id='viz'></a>
## 8. Visualisation: rankings and volcano plot

Standard visualisations for E-distance results:

1. **Ranked bar chart** — perturbations ordered by E-distance, coloured by significance
2. **Volcano plot** — E-distance (x-axis) vs. −log₁₀(p-value) (y-axis)  
   This is conceptually analogous to a volcano plot in differential expression, but with
   E-distance as the effect size rather than fold-change.
3. **Pairwise distance heatmap** — top N perturbations; reveals clusters of similar programs

In [ ]:
# ---- 8.1  Ranked bar chart --------------------------------------------------
df_plot = df_etest.sort_values('edistance', ascending=True).copy()
colors  = ['tomato' if sig else 'steelblue' for sig in df_plot['significant']]

fig, ax = plt.subplots(figsize=(5, max(4, len(df_plot) * 0.25)))
bars = ax.barh(df_plot['perturbation'], df_plot['edistance'],
               color=colors, height=0.7, edgecolor='none')

from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='tomato',    label='FDR < 0.05 (significant)'),
    Patch(facecolor='steelblue', label='FDR ≥ 0.05 (not significant)'),
]
ax.legend(handles=legend_elements, fontsize=8, loc='lower right')
ax.set_xlabel('E-distance to control (PCA space)')
ax.set_title('Perturbation effect sizes — NormanWeissman2019\n(single perturbations, ≥200 cells)')
ax.axvline(0, color='black', lw=0.5)
plt.tight_layout()
plt.savefig(FIG_DIR / '00b_edist_ranked.pdf', bbox_inches='tight')
plt.show()

In [ ]:
# ---- 8.2  Volcano-style plot (E-distance vs -log10 adjusted p-value) -------
# Replace adjusted p-values of 0 with a small floor to avoid log(0)
df_plot2 = df_etest.copy()
df_plot2['pvalue_adj'] = df_plot2['pvalue_adj'].clip(lower=1e-6)
df_plot2['neg_log10_p'] = -np.log10(df_plot2['pvalue_adj'])

FDR_THRESHOLD = 0.05
yline = -np.log10(FDR_THRESHOLD)

sig_mask  = df_plot2['significant']
nsig_mask = ~sig_mask

fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(df_plot2.loc[nsig_mask, 'edistance'],
           df_plot2.loc[nsig_mask, 'neg_log10_p'],
           s=30, alpha=0.5, color='steelblue', label='FDR ≥ 0.05')
ax.scatter(df_plot2.loc[sig_mask, 'edistance'],
           df_plot2.loc[sig_mask, 'neg_log10_p'],
           s=40, alpha=0.8, color='tomato', label='FDR < 0.05', zorder=3)

# Label the top 5 hits
top5 = df_plot2.nlargest(5, 'edistance')
for _, row in top5.iterrows():
    ax.annotate(
        row['perturbation'],
        xy=(row['edistance'], row['neg_log10_p']),
        xytext=(5, 2), textcoords='offset points',
        fontsize=7, color='black',
    )

ax.axhline(yline, color='gray', ls='--', lw=1, label=f'FDR = {FDR_THRESHOLD}')
ax.set_xlabel('E-distance to control')
ax.set_ylabel('−log₁₀(BH-adjusted p-value)')
ax.set_title('E-distance volcano plot — NormanWeissman2019')
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(FIG_DIR / '00b_edist_volcano.pdf', bbox_inches='tight')
plt.show()

In [ ]:
# ---- 8.3  Pairwise E-distance heatmap for top N perturbations --------------
# Shows which perturbations produce *similar* transcriptional programs
# (low pairwise E-distance = similar program; high = distinct)

TOP_N = min(20, len(pert_pass))
top_perts = df_etest.nlargest(TOP_N, 'edistance')['perturbation'].tolist()
all_groups = top_perts + ['control']

# Compute all pairwise E-distances among the top perturbations + control
pairwise_matrix = pd.DataFrame(0.0, index=all_groups, columns=all_groups)

rng_pw = np.random.default_rng(7)
SUBSAMPLE_PW = 300   # subsample each group for tractable pairwise computation

group_embeddings = {}
for g in all_groups:
    idx = np.where(labels == g)[0]
    if len(idx) > SUBSAMPLE_PW:
        idx = rng_pw.choice(idx, size=SUBSAMPLE_PW, replace=False)
    group_embeddings[g] = X_pca[idx]

for i, g1 in enumerate(all_groups):
    for j, g2 in enumerate(all_groups):
        if i < j:
            e = compute_edist(group_embeddings[g1], group_embeddings[g2])
            pairwise_matrix.loc[g1, g2] = e
            pairwise_matrix.loc[g2, g1] = e   # symmetric

fig, ax = plt.subplots(figsize=(10, 9))
sns.heatmap(
    pairwise_matrix.astype(float),
    cmap='viridis', ax=ax,
    xticklabels=True, yticklabels=True,
    linewidths=0.3, linecolor='white',
    cbar_kws={'label': 'E-distance'},
)
ax.set_title(
    f'Pairwise E-distances — top {TOP_N} perturbations + control\n'
    '(dark = similar programs; bright = distinct programs)'
)
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.yticks(rotation=0, fontsize=8)
plt.tight_layout()
plt.savefig(FIG_DIR / '00b_pairwise_heatmap.pdf', bbox_inches='tight')
plt.show()

<a id='qc'></a>
## 9. QC: Cell-count requirements and estimate stability

The E-distance estimator converges at rate $O(1/\sqrt{n})$.  For small perturbation groups
(< 50 cells), the within-group term $\mathbb{E}[\|X - X'\|]$ is very noisy and E-distance
estimates can be *inflated* — a null perturbation may appear to have a large effect.

The scPerturb paper recommends **≥ 200 cells** per perturbation for reliable E-test results.

Below we demonstrate this by bootstrapping — repeatedly subsampling from a large perturbation
group at different cell counts and measuring the variance of the E-distance estimate.

In [ ]:
# ---- 9.1  Bootstrap E-distance stability vs. cell count --------------------
# Use the highest-effect perturbation (stable enough to have a clear true value)
target_pert = df_etest.iloc[0]['perturbation']
pert_idx    = np.where(labels == target_pert)[0]
X_target    = X_pca[pert_idx]   # all cells for this perturbation

cell_counts  = [20, 50, 100, 200, 400, len(X_target)]
n_bootstrap  = 50   # bootstrap replicates per cell-count level
rng_boot     = np.random.default_rng(123)

boot_results = []
for n_cells in cell_counts:
    if n_cells > len(X_target):
        continue
    for _ in range(n_bootstrap):
        sub_pert = rng_boot.choice(len(X_target), size=n_cells, replace=False)
        sub_ctrl = rng_boot.choice(len(X_ctrl),   size=min(n_cells, len(X_ctrl)), replace=False)
        e = compute_edist(X_target[sub_pert], X_ctrl[sub_ctrl])
        boot_results.append({'n_cells': n_cells, 'edistance': e})

df_boot = pd.DataFrame(boot_results)

# Plot bootstrap distributions
fig, ax = plt.subplots(figsize=(8, 4))
groups = df_boot.groupby('n_cells')['edistance']
positions = sorted(df_boot['n_cells'].unique())
data_to_plot = [groups.get_group(n).values for n in positions]

bp = ax.boxplot(
    data_to_plot,
    positions=range(len(positions)),
    widths=0.5,
    patch_artist=True,
    boxprops=dict(facecolor='steelblue', alpha=0.6),
    medianprops=dict(color='tomato', lw=2),
)

ax.set_xticks(range(len(positions)))
ax.set_xticklabels(positions)
ax.axvline(positions.index(200) - 0.5 + 0.5, color='orange', ls='--', lw=1.5,
           label='Recommended minimum (200 cells)') if 200 in positions else None
ax.set_xlabel('Cells per perturbation (subsampled)')
ax.set_ylabel('E-distance estimate')
ax.set_title(f'Bootstrap E-distance stability\nPerturbation: {target_pert}  |  n_bootstrap = {n_bootstrap}')
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(FIG_DIR / '00b_bootstrap_stability.pdf', bbox_inches='tight')
plt.show()

print('Bootstrap CV (coefficient of variation) by cell count:')
for n_cells in positions:
    vals = groups.get_group(n_cells)
    cv   = vals.std() / vals.mean() if vals.mean() > 0 else float('nan')
    print(f'  n = {n_cells:4d} cells  →  mean = {vals.mean():.3f},  CV = {cv:.1%}')

<a id='summary'></a>
## 10. Summary and key takeaways

### What we implemented

| Component | Description | Location |
|-----------|-------------|----------|
| `compute_edist(X_pert, X_ctrl)` | Sample E-distance (3-term formula) | This notebook / `src/perturbation_analysis/edistance.py` |
| `compute_estatistic(...)` | Sample-size-scaled test statistic | This notebook |
| `run_etest(...)` | Permutation-based E-test with p-value | This notebook |
| BH FDR correction | Multiple-testing correction for screens | `statsmodels.stats.multitest.multipletests` |
| `pt.tl.Distance` | Production E-distance API | `pertpy` |

### Key formulae

$$\hat{E}(P, Q) = 2 \cdot \overline{d}(X, Y) - \overline{d}(X, X) - \overline{d}(Y, Y)$$

$$T = \frac{nm}{2(n+m)} \cdot \hat{E}(P, Q) \quad \text{(test statistic)}$$

$$p = \frac{\#\{T_b \geq T_{\text{obs}}\} + 1}{B + 1} \quad \text{(permutation p-value)}$$

### Practical guidelines

| Guideline | Value / Rule |
|-----------|-------------|
| Minimum cells for reliable E-distance | **200** per perturbation |
| Minimum cells for reliable E-test | **200** per perturbation |
| Representation space | **50 PCA components** on 2000 HVGs |
| Number of permutations | 1,000 (exploration) / 10,000 (final) |
| Multiple-testing correction | Benjamini-Hochberg FDR 5% |
| Large control groups | Subsample to ~2,000 for speed |

### What comes next

- **Phase 2 notebooks** (`01_replogle_2022/`) apply this pipeline at genome scale:
  ~10,000 perturbations across RPE1 and K562 cells.
- **Cross-dataset comparison** (`cross_dataset/`) uses pairwise E-distances to identify
  perturbations with *conserved* effects across cardiac iPSC-CM datasets.
- **`src/perturbation_analysis/edistance.py`** wraps these functions for reuse across all
  subsequent analysis notebooks.

### References

- Székely, G.J. & Rizzo, M.L. (2004). Testing for equal distributions in high dimension.
  *InterStat* Nov(5).
- Peidli, S. et al. (2024). scPerturb: harmonized single-cell perturbation data.
  *Nature Methods* 21, 531–540. https://doi.org/10.1038/s41592-023-02144-y
- Phipson, B. & Smyth, G.K. (2010). Permutation P-values should never be zero.
  *Statistical Applications in Genetics and Molecular Biology* 9(1).

In [ ]:
from datetime import datetime
import subprocess

timestamp   = datetime.now().strftime('%Y%m%d_%H%M')
report_dir  = Path('../../results/reports')
report_dir.mkdir(parents=True, exist_ok=True)
report_path = report_dir / f'00b_edistance_tutorial_{timestamp}.html'

nb_path = (
    __vsc_ipynb_file__
    if '__vsc_ipynb_file__' in dir()
    else '00b_edistance_tutorial.ipynb'
)

subprocess.run(
    ['jupyter', 'nbconvert', '--to', 'html', '--output', str(report_path), nb_path],
    check=False,
)
print(f'Report saved to {report_path}')